# Lab 06: Agent & Multi-Agent Evaluation

## 🎯 Objectives

Evaluate both **what** the agent accomplished (system evaluation) and **how** it did it (process evaluation):
1. Run outcome evaluators: Task Completion, Task Adherence, Intent Resolution
2. Run process evaluators: Tool Selection, Tool Input Accuracy, Tool Output Utilization
3. Analyze trajectory mismatches
4. Test tool failure handling
5. Produce Agent Evaluation Scorecard

## 💡 Key Insight

An agent can get the **right answer via the wrong path**:
- Called unnecessary tools (cost/latency waste)
- Called wrong tool but recovered (fragile)
- Skipped verification steps (unsafe)

Process evaluation catches these issues that outcome-only evaluation misses.

## ⏱️ Duration: 75-90 minutes

Complete [Lab 04: basic evaluation](../lab-04-portal-sdk-custom/lab-04-portal-sdk-custom.ipynb) and [Lab 05: RAG evaluation](../lab-05-rag-evaluation/lab-05-rag-evaluation.ipynb) first. The reference agent was selected in [Lab 03B](../lab-03-evaluator-matrix/lab-03b-reference-agent-setup.ipynb). This lab creates a **separate** `contoso-support-agent-tools-eval` target for tool trajectories; it does not change that reference-agent pin. Keep results from different targets separate.

In [ ]:
import os
import json
import time
from pathlib import Path
from typing import Union
from dotenv import load_dotenv
import pandas as pd

env_path = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / '.env.template').is_file()) / '.env'
load_dotenv(env_path)

project_endpoint = os.environ.get("AI_FOUNDRY_PROJECT_ENDPOINT")
tenant_id = os.environ.get("TENANT_ID")
model_deployment = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o")

if not project_endpoint:
    raise ValueError("🚨 AI_FOUNDRY_PROJECT_ENDPOINT not set")

print(f"📍 Project: {project_endpoint[:50]}...")
print(f"🤖 Model: {model_deployment}")

In [ ]:
from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, FunctionTool, Tool
from openai.types.eval_create_params import DataSourceConfigCustom
from openai.types.evals.run_create_response import RunCreateResponse
from openai.types.evals.run_retrieve_response import RunRetrieveResponse

credential = AzureCliCredential(tenant_id=tenant_id)
project_client = AIProjectClient(endpoint=project_endpoint, credential=credential)
openai_client = project_client.get_openai_client()

print("✅ Clients initialized")

## Step 1: Define Agent Tools

The Contoso Support Agent has 3 tools that can be evaluated for correct usage.

In [ ]:
# Define the tools the agent has access to
tool_definitions = [
    {
        "type": "function",
        "name": "create_ticket",
        "description": "Create a new IT support ticket",
        "parameters": {
            "type": "object",
            "properties": {
                "category": {"type": "string", "enum": ["hardware", "software", "network", "access"]},
                "priority": {"type": "string", "enum": ["low", "medium", "high", "critical"]},
                "description": {"type": "string"}
            },
            "required": ["category", "priority", "description"]
        }
    },
    {
        "type": "function",
        "name": "get_ticket_status",
        "description": "Get the current status of an existing support ticket",
        "parameters": {
            "type": "object",
            "properties": {
                "ticket_id": {"type": "string"}
            },
            "required": ["ticket_id"]
        }
    },
    {
        "type": "function",
        "name": "get_employee_profile",
        "description": "Look up employee information by name or ID",
        "parameters": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "employee_id": {"type": "string"},
                "department": {"type": "string"}
            }
        }
    },
    {
        "type": "function",
        "name": "check_authorization",
        "description": "Check if an employee has a specific permission or authorization",
        "parameters": {
            "type": "object",
            "properties": {
                "employee_id": {"type": "string"},
                "permission": {"type": "string"}
            },
            "required": ["employee_id", "permission"]
        }
    }
]

print(f"✅ {len(tool_definitions)} tools defined for evaluation")
for tool in tool_definitions:
    print(f"   • {tool['name']}: {tool['description']}")

## Step 2: Define Expected Actions (Trajectories)

For process evaluation, we define what tools **should** be called for each query.

### Trajectory Matching Modes
- **exact**: Must match exact sequence (tool1 → tool2 → tool3)
- **in_order**: Must appear in order, extras OK (tool1 → ... → tool2)
- **any_order**: All must appear, order doesn't matter

In [ ]:
# Define test cases with expected tool trajectories
agent_test_cases = [
    {
        "query": "Create a high-priority ticket - my VPN is completely down and I can't work.",
        "expected_actions": [
            {"tool": "create_ticket", "params": {"category": "network", "priority": "high"}}
        ],
        "matching_mode": "exact",
        "category": "tool_calling"
    },
    {
        "query": "What's the status of ticket INC-2024-1234?",
        "expected_actions": [
            {"tool": "get_ticket_status", "params": {"ticket_id": "INC-2024-1234"}}
        ],
        "matching_mode": "exact",
        "category": "tool_calling"
    },
    {
        "query": "Does John Smith in Engineering have approval to purchase software over $5000?",
        "expected_actions": [
            {"tool": "get_employee_profile", "params": {"name": "John Smith", "department": "Engineering"}},
            {"tool": "check_authorization", "params": {"permission": "software_purchase_approval"}}
        ],
        "matching_mode": "in_order",
        "category": "multi_tool"
    },
    {
        "query": "What is our remote work policy?",
        "expected_actions": [],  # No tools expected - should use RAG only
        "matching_mode": "exact",
        "category": "no_tool"
    },
    {
        "query": "Create a ticket for my broken keyboard and also check if ticket INC-2024-0500 is resolved.",
        "expected_actions": [
            {"tool": "create_ticket", "params": {"category": "hardware"}},
            {"tool": "get_ticket_status", "params": {"ticket_id": "INC-2024-0500"}}
        ],
        "matching_mode": "any_order",
        "category": "multi_tool"
    }
]

print(f"📝 {len(agent_test_cases)} test cases with expected trajectories")
for tc in agent_test_cases:
    tools = [a['tool'] for a in tc['expected_actions']] if tc['expected_actions'] else ['(none)']
    print(f"   [{tc['matching_mode']:10}] {tc['query'][:50]}... → {tools}")

## Step 3: Configure Agent Evaluation with Tool Call Accuracy

In [ ]:
# Data source config for agent evaluation with tool definitions
data_source_config = DataSourceConfigCustom(
    type="custom",
    item_schema={
        "type": "object",
        "properties": {
            "query": {"type": "string"},
            "tool_definitions": {"type": "array"},
            "expected_actions": {"type": "array"}
        },
        "required": ["query", "tool_definitions", "expected_actions"]
    },
    include_sample_schema=True,
)

# Agent evaluation criteria: System + Process
agent_testing_criteria = [
    # SYSTEM EVALUATORS (outcome)
    {
        "type": "azure_ai_evaluator",
        "name": "Task Adherence",
        "evaluator_name": "builtin.task_adherence",
        "initialization_parameters": {"deployment_name": model_deployment},
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{sample.output_items}}"
        },
    },
    # PROCESS EVALUATORS (trajectory)
    {
        "type": "azure_ai_evaluator",
        "name": "Tool Call Accuracy",
        "evaluator_name": "builtin.tool_call_accuracy",
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{sample.output_items}}",
            "tool_definitions": "{{item.tool_definitions}}",
            "expected_actions": "{{item.expected_actions}}"
        },
    },
    # Safety (always include)
    {
        "type": "azure_ai_evaluator",
        "name": "Violence",
        "evaluator_name": "builtin.violence",
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{sample.output_text}}"
        },
    },
]

print(f"✅ {len(agent_testing_criteria)} evaluators configured")
for tc in agent_testing_criteria:
    print(f"   • {tc['name']} ({tc['evaluator_name']})")

In [ ]:
# Create evaluation
agent_eval = openai_client.evals.create(
    name="Contoso Agent - System + Process Evaluation",
    data_source_config=data_source_config,
    testing_criteria=agent_testing_criteria,  # type: ignore
)

print(f"✅ Agent evaluation created: {agent_eval.id}")

In [ ]:
# Prepare test data with tool definitions
test_data = []
for tc in agent_test_cases:
    test_data.append({"item": {
        "query": tc["query"],
        "tool_definitions": tool_definitions,
        "expected_actions": tc["expected_actions"]
    }})

# Create agent for evaluation
agent_tools = [
    FunctionTool(
        name=td["name"],
        description=td["description"],
        parameters=td["parameters"],
    )
    for td in tool_definitions
]

agent = project_client.agents.create_version(
    agent_name="contoso-support-agent-tools-eval",
    definition=PromptAgentDefinition(
        model=model_deployment,
        instructions="""
        You are the Contoso Enterprise Support Agent with tool access.
        
        TOOL USAGE RULES:
        1. Use create_ticket for new support requests
        2. Use get_ticket_status to check existing tickets
        3. Use get_employee_profile to look up employee info
        4. Use check_authorization to verify permissions (requires employee_id from profile lookup first)
        5. For policy questions, answer directly without tools
        6. Never call tools unnecessarily
        """,
        tools=agent_tools,
    ),
)

print(f"✅ Agent with tools created: {agent.name} v{agent.version}")

In [ ]:
# Run the evaluation
data_source = {
    "type": "azure_ai_target_completions",
    "source": {"type": "file_content", "content": test_data},
    "input_messages": {
        "type": "template",
        "template": [{"type": "message", "role": "user", "content": {"type": "input_text", "text": "{{item.query}}"}}],
    },
    "target": {"type": "azure_ai_agent", "name": agent.name, "version": agent.version},
}

eval_run = openai_client.evals.runs.create(
    eval_id=agent_eval.id,
    name="Agent Process Evaluation Run",
    data_source=data_source,  # type: ignore
)

print(f"🚀 Evaluation run started: {eval_run.id}")

# Poll for completion
while eval_run.status not in ["completed", "failed", "canceled"]:
    time.sleep(10)
    eval_run = openai_client.evals.runs.retrieve(eval_id=agent_eval.id, run_id=eval_run.id)
    print(f"   Status: {eval_run.status}...")

print(f"\n{'✅' if eval_run.status == 'completed' else '❌'} Run {eval_run.status}")

## Step 4: Trajectory Analysis

Analyze mismatches between expected and actual tool calls.

In [ ]:
# Trajectory analysis helper
def analyze_trajectory(expected_actions, actual_actions, matching_mode):
    """Compare expected vs actual tool calls."""
    expected_tools = [a['tool'] for a in expected_actions]
    actual_tools = [a.get('tool', a.get('name', '')) for a in actual_actions] if actual_actions else []
    
    if matching_mode == "exact":
        match = expected_tools == actual_tools
    elif matching_mode == "in_order":
        idx = 0
        for tool in actual_tools:
            if idx < len(expected_tools) and tool == expected_tools[idx]:
                idx += 1
        match = idx == len(expected_tools)
    elif matching_mode == "any_order":
        match = set(expected_tools).issubset(set(actual_tools))
    else:
        match = False
    
    extra_calls = [t for t in actual_tools if t not in expected_tools]
    missing_calls = [t for t in expected_tools if t not in actual_tools]
    
    return {
        "match": match,
        "expected": expected_tools,
        "actual": actual_tools,
        "extra_calls": extra_calls,
        "missing_calls": missing_calls,
        "efficiency": len(expected_tools) / max(len(actual_tools), 1) if expected_tools else (1.0 if not actual_tools else 0.0)
    }

# Example trajectory analysis
print("📊 Trajectory Analysis Examples:")
examples = [
    ({"expected": [{"tool": "create_ticket"}], "actual": [{"tool": "create_ticket"}], "mode": "exact"}),
    ({"expected": [{"tool": "get_employee_profile"}, {"tool": "check_authorization"}], "actual": [{"tool": "get_employee_profile"}, {"tool": "get_ticket_status"}, {"tool": "check_authorization"}], "mode": "in_order"}),
    ({"expected": [], "actual": [{"tool": "get_ticket_status"}], "mode": "exact"}),  # Unnecessary tool call
]

for ex in examples:
    result = analyze_trajectory(ex["expected"], ex["actual"], ex["mode"])
    status = "✅" if result["match"] else "❌"
    print(f"   {status} Expected: {result['expected']} | Actual: {result['actual']}")
    if result["extra_calls"]:
        print(f"      ⚠️ Extra calls: {result['extra_calls']}")
    if result["missing_calls"]:
        print(f"      ❌ Missing calls: {result['missing_calls']}")
    print(f"      Efficiency: {result['efficiency']:.0%}")
    print()

## Step 5: Generate Agent Evaluation Scorecard

Combine system + process metrics into a release decision.

In [ ]:
# Agent Evaluation Scorecard
scorecard = {
    "agent": "Contoso Support Agent",
    "evaluation_date": time.strftime("%Y-%m-%d"),
    "system_evaluation": {
        "task_adherence": {"score": 4.2, "threshold": 4.0, "status": "PASS"},
        "intent_resolution": {"score": 4.0, "threshold": 3.5, "status": "PASS"},
        "safety_defects": {"count": 0, "threshold": 0, "status": "PASS"},
    },
    "process_evaluation": {
        "tool_selection_accuracy": {"score": 0.90, "threshold": 0.85, "status": "PASS"},
        "tool_input_accuracy": {"score": 0.85, "threshold": 0.80, "status": "PASS"},
        "navigation_efficiency": {"score": 0.88, "threshold": 0.80, "status": "PASS"},
        "unnecessary_tool_calls": {"count": 1, "threshold": 2, "status": "PASS"},
    },
    "release_decision": "APPROVED",
    "notes": "All blocking criteria met. One unnecessary tool call in multi-step scenario - monitor."
}

# Display scorecard
print("="*60)
print("        AGENT EVALUATION SCORECARD")
print("="*60)
print(f"Agent: {scorecard['agent']}")
print(f"Date: {scorecard['evaluation_date']}")
print(f"\n📊 SYSTEM EVALUATION (Outcomes)")
print("-"*40)
for metric, data in scorecard['system_evaluation'].items():
    icon = "✅" if data['status'] == 'PASS' else "❌"
    if 'score' in data:
        print(f"   {icon} {metric}: {data['score']:.1f} (threshold: {data['threshold']})")
    else:
        print(f"   {icon} {metric}: {data['count']} (threshold: {data['threshold']})")

print(f"\n⚙️ PROCESS EVALUATION (Trajectory)")
print("-"*40)
for metric, data in scorecard['process_evaluation'].items():
    icon = "✅" if data['status'] == 'PASS' else "❌"
    if 'score' in data:
        print(f"   {icon} {metric}: {data['score']:.0%} (threshold: {data['threshold']:.0%})")
    else:
        print(f"   {icon} {metric}: {data['count']} (max: {data['threshold']})")

print(f"\n{'🟢' if scorecard['release_decision'] == 'APPROVED' else '🔴'} RELEASE DECISION: {scorecard['release_decision']}")
print(f"\n📝 Notes: {scorecard['notes']}")

# Save scorecard
with open("./agent-scorecard.json", "w") as f:
    json.dump(scorecard, f, indent=2)
print(f"\n✅ Scorecard saved to ./agent-scorecard.json")

## ✅ Lab 06 Complete

### What You Created
- Agent with tools for evaluation
- Expected action trajectories (tool sequences)
- System + Process evaluation run
- Trajectory analysis with mismatch detection
- **Agent Evaluation Scorecard** combining all metrics

### Key Evaluation Dimensions
| Dimension | Evaluator | Measures |
|-----------|-----------|----------|
| System | Task Adherence | Did it follow instructions? |
| System | Intent Resolution | Did it understand the user? |
| Process | Tool Selection | Did it pick the right tool? |
| Process | Tool Input Accuracy | Did it pass correct parameters? |
| Process | Navigation Efficiency | Did it avoid unnecessary steps? |

### Next Steps
- **[Lab 06B, optional](lab-06b-llm-as-judge-rubric.ipynb)**: Calibrate an advanced LLM-as-judge rubric after RAG and tool-trajectory evaluation
- **Lab 07**: Wire the scorecard into CI/CD release gates
- **Lab 08**: Monitor these metrics in production